# OSM: servicios de emergencia

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad de análisis

Archivo: `data/externos/osm_emergencia.geojson`. Una fila representa un objeto OSM de policía o bomberos.
La fecha de descarga no equivale a fecha de verificación del objeto en terreno.

In [ ]:
PATH = ROOT / 'data/externos/osm_emergencia.geojson'
meta = ficha(PATH)
raw, df, g = leer_geojson(PATH)
print("Archivo:", meta["archivo"], "| filas:", len(df), "| propiedades:", len(df.columns))
display(pd.Series(meta, name="procedencia").to_frame())

## 2. Calidad por columna y geometría

In [ ]:
calidad = mostrar_perfil(df)
geo = geometria(g)
display(pd.Series(geo, name="geometria").to_frame())
resumen = {"dataset": '05_osm_emergencia', **meta, "filas": len(df), "columnas": list(df.columns),
           "calidad_columnas": calidad.to_dict("records"), "geometria": geo}

## 3. Separación de funciones y atributos disponibles

In [ ]:
tipos=conteos(df.amenity,"servicio")
display(tipos)
display(faltantes(df,["osm_id","name","phone","opening_hours","operational_status","capacity","check_date"]))
grafico_barras(tipos,"Policía y bomberos tienen funciones diferentes", "05_tipos")
resumen.update({"tipos":df.amenity.value_counts().to_dict(),"osm_id_duplicados":int(df.osm_id.duplicated().sum())})

## Reglas decididas para plata

- **Incorporar bomberos al análisis de respuesta**, conservando policía como categoría separada.
- Usar una tabla de servicios con `tipo_servicio`; la distancia a bomberos se calcula solo con `fire_station`.
- No sumar policía y bomberos como si fueran brigadas equivalentes. No deducir cantidad de vehículos, personal u operatividad de su presencia en OSM.
- Preservar identificadores, fecha de corte y faltantes operativos.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('05_osm_emergencia', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")